# Notebook kiểm tra project ConvNeXt-Tiny và ResNet18

Mở notebook từ thư mục gốc project và chạy từng cell theo thứ tự.

Notebook kiểm tra môi trường/GPU, forward hai model pretrained và một epoch training smoke với dữ liệu giả. Cell tải CIFAR-10 và cell huấn luyện thật được tắt mặc định để tránh download hoặc chạy dài ngoài ý muốn. Chỉ bật chúng khi đã sẵn sàng.


In [1]:
from pathlib import Path
import os
import sys
import torch
import torchvision

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "src" / "cifar_compare" / "train.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Không tìm thấy thư mục gốc project trong working directory.")
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA build:", torch.version.cuda)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Project root: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification
Python: 3.14.4
PyTorch: 2.15.0.dev20260926+cu130
Torchvision: 0.30.0.dev20260928+cu130
CUDA available: True
CUDA build: 13.0
Device: NVIDIA GeForce RTX 5070 Ti


## 1. Kiểm tra model factory và pretrained weights

Lần đầu chạy có thể tải weights từ torchvision. Cell này chỉ chạy inference trên tensor giả, không tải CIFAR-10 và không huấn luyện.


In [2]:
from src.cifar_compare.models import build_model, count_parameters

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_SIZE = 224
NUM_CLASSES = 10

for model_name in ("resnet18", "convnext_tiny"):
    model = build_model(
        model_name,
        num_classes=NUM_CLASSES,
        pretrained=True,
        weights_name="IMAGENET1K_V1",
    ).to(DEVICE).eval()

    sample = torch.zeros(2, 3, INPUT_SIZE, INPUT_SIZE, device=DEVICE)
    with torch.inference_mode():
        logits = model(sample)

    assert tuple(logits.shape) == (2, NUM_CLASSES), tuple(logits.shape)
    assert torch.isfinite(logits).all().item()
    print(model_name, "| shape:", tuple(logits.shape),
          "| finite: True | parameters:", count_parameters(model))

    del model, sample, logits
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


resnet18 | shape: (2, 10) | finite: True | parameters: 11181642
convnext_tiny | shape: (2, 10) | finite: True | parameters: 27827818


## 2. Kiểm tra training loop bằng dữ liệu giả

Chạy một epoch với bốn tensor ngẫu nhiên và ResNet18 không pretrained. Việc này kiểm tra forward/backward, optimizer, validation, metrics và checkpoint. Accuracy không phải kết quả CIFAR-10.


In [3]:
from datetime import datetime
from torch.utils.data import DataLoader, TensorDataset

from src.cifar_compare.train import DEFAULTS, train_model
from src.cifar_compare.utils import resolve_device

smoke_config = dict(DEFAULTS)
smoke_config.update({
    "model": "resnet18",
    "pretrained": False,
    "epochs": 1,
    "batch_size": 2,
    "num_workers": 0,
    "amp": True,
    "early_stopping_patience": 0,
    "output_dir": "artifacts/notebook_smoke",
    "run_name": "resnet18_" + datetime.now().strftime("%Y%m%d_%H%M%S"),
})

images = torch.randn(4, 3, INPUT_SIZE, INPUT_SIZE)
labels = torch.tensor([0, 1, 2, 3])
loader = DataLoader(TensorDataset(images, labels), batch_size=2)

summary = train_model(
    "resnet18",
    smoke_config,
    loader,
    loader,
    ["airplane", "automobile", "bird", "cat", "deer",
     "dog", "frog", "horse", "ship", "truck"],
    device=resolve_device(smoke_config["device"]),
)
print(summary)

assert Path(summary["best_checkpoint"]).is_file()
assert Path(summary["last_checkpoint"]).is_file()
assert Path(summary["metrics_csv"]).is_file()
print("Training smoke check passed.")


{'model': 'resnet18', 'best_epoch': 1, 'best_val_accuracy': 0.25, 'best_val_loss': 2.0576171875, 'epochs_completed': 1, 'best_checkpoint': 'artifacts\\notebook_smoke\\checkpoints\\resnet18_20261103_014953\\best.pt', 'last_checkpoint': 'artifacts\\notebook_smoke\\checkpoints\\resnet18_20261103_014953\\last.pt', 'metrics_csv': 'artifacts\\notebook_smoke\\logs\\resnet18_20261103_014953\\metrics.csv', 'device': 'cuda'}
Training smoke check passed.


## 3. Tùy chọn: kiểm tra CIFAR-10

Mặc định cell sau không tải dữ liệu. Đổi cờ thành True nếu muốn kiểm tra data.py; lần đầu có thể tải CIFAR-10 và lưu split indices.


In [4]:
RUN_CIFAR_DATA_CHECK = True
cifar_loaders = None
class_names = None

if RUN_CIFAR_DATA_CHECK:
    from src.cifar_compare.data import get_dataloaders

    data_config = dict(DEFAULTS)
    data_config.update({
    "input_size": 512,
    "batch_size": 256,
    "num_workers": 4,
    "device": "cuda",
    })
    train_loader, val_loader, test_loader, class_names = get_dataloaders(data_config)
    cifar_loaders = (train_loader, val_loader, test_loader)

    batch_images, batch_labels = next(iter(train_loader))
    print("Samples train/val/test:",
          len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
    print("Classes:", class_names)
    print("Batch:", tuple(batch_images.shape), tuple(batch_labels.shape))
    assert batch_images.shape[1:] == (3, data_config["input_size"], data_config["input_size"])
    assert torch.isfinite(batch_images).all().item()
else:
    print("Skipped. Set RUN_CIFAR_DATA_CHECK = True to download/check CIFAR-10.")


  0%|          | 197k/170M [00:06<1:29:00, 31.9kB/s] 


KeyboardInterrupt: 

## 4. Tùy chọn: huấn luyện thật

Chỉ bật sau khi DataLoader đã chạy đạt và nhóm sẵn sàng dùng GPU/thời gian. Cấu hình mặc định chạy tối đa 10 epoch và có early stopping.


In [5]:
RUN_FULL_TRAINING = False
MODEL_TO_TRAIN = "resnet18"  # Chạy lại với convnext_tiny cho model còn lại.

if RUN_FULL_TRAINING:
    if cifar_loaders is None:
        raise RuntimeError("Hãy bật cell kiểm tra CIFAR-10 trước.")
    train_loader, val_loader, _test_loader = cifar_loaders
    final_config = dict(DEFAULTS)
    final_config["run_name"] = (
        MODEL_TO_TRAIN + "_" + datetime.now().strftime("%Y%m%d_%H%M%S")
    )
    result = train_model(
        MODEL_TO_TRAIN, final_config, train_loader, val_loader, class_names,
        device=resolve_device(final_config["device"]),
    )
    print(result)
else:
    print("Skipped. Set RUN_FULL_TRAINING = True only when ready.")


Skipped. Set RUN_FULL_TRAINING = True only when ready.
